In [1]:
import csv
import nibabel as nib
import matplotlib.pyplot as plt
import random
import torch
import os
import numpy as np
from model1 import CNN_3D,NiiDataset,MultiModalTransformer,NeuralNet,KGMultiModalTransformer
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import nibabel as nib
import shutil
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import math
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd

In [2]:
#生物标志物 ad 90*48   no 349*58
path_existence = []
data_normal=[]
data_ad=[]
data_mci=[]
count_ad=0
count_no=0
count_mci=0
with open('normal.csv', mode='r', newline='', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)  
    for row in csv_reader:
        path = 'normal_nii_kg/' + row[1]
        exists = os.path.exists(path)
        path_existence.append((path, exists))
        if exists:
            count_no=count_no+1
            data_normal.append(row)
            
with open('AD.csv', mode='r', newline='', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader) 
    for row in csv_reader:
        path = 'ad_nii_KG/' + row[1]
        exists = os.path.exists(path)
        path_existence.append((path, exists))
        if exists:
            count_ad=count_ad+1
            data_ad.append(row)
            
with open('mci.csv', mode='r', newline='', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader) 
    for row in csv_reader:
        path = 'NC_nii_kg/' + row[1]
        exists = os.path.exists(path)
        path_existence.append((path, exists))
        if exists:
            count_mci=count_mci+1
            data_mci.append(row)
print(count_ad) #151
print(count_no) #211
print(count_mci) #475

151
211
475


In [3]:
import torch
import torch.nn as nn
import numpy as np
import math

# 基础替换字典
replace_dict = {'female': '0', 'male': '1', 'whi': '0', 'blk': '1', '': '0','ind':'3','ans':'4'}

# 隔离诊断标签的列索引提取规则
# EHR (14维): age(2), gender(3), education(4), hispanic(5), race(6) + his_系列(57-65)
ehr_cols = [2, 3, 4, 5, 6, 57, 58, 59, 60, 61, 62, 63, 64, 65] 
# BIO (39维): apoe(7) + mmse/cdr(20-22) + gds/moca(66, 67) + trail/npiq/faq等(24-56)
bio_cols = [7, 20, 21, 22, 66, 67] + list(range(24, 57))

# 统一且安全的预处理函数
# 统一且安全的预处理函数
def preprocess_data(data, replace_dict, target_cols):
    processed_data = []
    for row in data:
        # 仅提取安全的特征列
        extracted_row = [row[idx] for idx in target_cols]
        # 字典映射替换
        extracted_row = [replace_dict.get(item, item) for item in extracted_row]
        
        numerical_row = []
        for item in extracted_row:
            try:
                val = float(item)
                # 拦截 NaN，保留为 np.nan 供后续专门的 Imputer 处理
                if math.isnan(val):
                    val = np.nan
            except ValueError:
                val = np.nan 
            numerical_row.append(val)
            
        processed_data.append(numerical_row)
    return np.array(processed_data)

# 加权算值函数 (保留你的原始定义)
def weighted_sum(tensor):
    weights = [0.2, 0.3, 0.5]
    weight_tensor = torch.tensor(weights, dtype=tensor.dtype, device=tensor.device)
    weighted_sum_result = torch.sum(tensor * weight_tensor, dim=1, keepdim=True)
    return weighted_sum_result

In [4]:
# 1. 提取 BIO 数组 (真实的 39 维特征)
ad_array = preprocess_data(data_ad, replace_dict, bio_cols)
normal_array = preprocess_data(data_normal, replace_dict, bio_cols)
mci_array = preprocess_data(data_mci, replace_dict, bio_cols)

# 2. 直接转换数据为 Tensor (无需外部线性层)
ad_tensor = torch.from_numpy(ad_array).float()
normal_tensor = torch.from_numpy(normal_array).float()
mci_tensor = torch.from_numpy(mci_array).float()

# 3. 打印结果 (预期输出类似: torch.Size([151, 39]))
print('AD_tensor shape:', ad_tensor.shape)      
print('Normal_tensor shape:', normal_tensor.shape) 
print('MCI_tensor shape:', mci_tensor.shape)

AD_tensor shape: torch.Size([151, 39])
Normal_tensor shape: torch.Size([211, 39])
MCI_tensor shape: torch.Size([475, 39])


In [5]:
# 1. 提取脱敏后的 EHR 数组 (内部已通过 replace_dict 转换为数值)
ad_data = preprocess_data(data_ad, replace_dict, ehr_cols)
normal_data = preprocess_data(data_normal, replace_dict, ehr_cols)
mci_data = preprocess_data(data_mci, replace_dict, ehr_cols)

# 2. 直接转换为 Tensor (无需外部线性层)
ad_EHR = torch.from_numpy(ad_data).float()
normal_EHR = torch.from_numpy(normal_data).float()
mci_EHR = torch.from_numpy(mci_data).float()

# 3. 打印结果 (预期输出类似: torch.Size([151, 14]))
print('ad.EHR--->', ad_EHR.shape)
print('normal.EHR--->', normal_EHR.shape)
print('mci.EHR--->', mci_EHR.shape)

ad.EHR---> torch.Size([151, 14])
normal.EHR---> torch.Size([211, 14])
mci.EHR---> torch.Size([475, 14])


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import torch
from torch.utils.data import DataLoader
import os
import pandas as pd

# 1. 实例化模型，将输出维度设为64，以严格匹配 Transformer 的 img_dim=64
from pathlib import Path
import sys
repo_root = Path.cwd() if (Path.cwd() / "experiments" / "kg_ablation.py").exists() else Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from experiments.kg_ablation import seed_everything
# Use identical fixed CNN features across KG variants; run seeds vary the classifier.
seed_everything(2026)

nii = CNN_3D(num_class=64)
nii = nii.to(device)
nii.eval() # 开启评估模式，冻结网络行为

def extract_mri_features(folder_path, csv_file, batch_size=16):
    dataset = NiiDataset(folder_path)
    filenames = pd.read_csv(csv_file)["filename"].astype(str).tolist()
    ordered_paths = [os.path.join(folder_path, name) for name in filenames]
    assert len(ordered_paths) == len(set(ordered_paths))
    assert set(ordered_paths) == set(dataset.file_list), "MRI files and CSV rows differ"
    dataset.file_list = ordered_paths  # Match EHR/KG row order exactly.
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_outputs = []
    
    # 2. 极其重要：关闭梯度计算，大幅节省显存并加快提取速度
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(dataloader):
            batch_data = batch_data.to(device)
            output = nii(batch_data)
            # 提取后立即转回 CPU，防止 GPU 显存堆积
            all_outputs.append(output.cpu())
            
    return torch.cat(all_outputs, dim=0)

all_ad = 'ad_nii_KG/'
all_normal = 'normal_nii_kg/'
all_mci = 'NC_nii_kg/'

# 3. 模块化调用
ad_output = extract_mri_features(all_ad, "AD.csv")
print('ad nii shape--->', ad_output.shape)

normal_output = extract_mri_features(all_normal, "normal.csv")
print('normal nii shape--->', normal_output.shape)

mci_output = extract_mri_features(all_mci, "mci.csv")
print('mci nii shape--->', mci_output.shape)

In [ ]:
from pathlib import Path
import sys
_asset_root = next(parent for parent in (Path.cwd(), *Path.cwd().parents)
                   if (parent / "experiments" / "config.py").exists())
if str(_asset_root) not in sys.path:
    sys.path.insert(0, str(_asset_root))
from experiments.config import resolve_graph_asset

import numpy as np
import torch
import pandas as pd
import re

# ================= 核心修改 1：对齐 CustomKG 的输出文件 =================
ENTITY_EMB_NPY = resolve_graph_asset('ADNI', 'CustomKG', 'CustomKG-ADNI.npy')
ENTITY2ID_FILE = resolve_graph_asset('ADNI', 'CustomKG', 'CustomKG-ADNI.txt')

# 1. 解析制表符分隔的 txt 字典
entity2id = {}
with open(ENTITY2ID_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            entity2id[parts[0]] = int(parts[1])

# 2. 加载嵌入矩阵 (此时维度自动匹配为 32)
emb = torch.from_numpy(np.load(ENTITY_EMB_NPY)).float()
embed_dim = emb.shape[1] 
max_seq_len = 35 

# 辅助函数：从文件名提取真实的 PTID，与 CustomKG 构建时的逻辑严格保持一致
def extract_ptid_from_filename(filename_str):
    match = re.search(r"(\d+_S_\d+)", str(filename_str))
    if match:
        return match.group(1)
    return str(filename_str).strip()

# ================= 核心修改 2：生成以患者自身节点为核心的图谱序列 =================
def get_patient_embedding_seq(filename: str):
    ptid = extract_ptid_from_filename(filename)
    
    vecs = []
    pad_masks = []
    
    # 如果该患者节点在 CustomKG 中存在
    if ptid in entity2id:
        # 第 0 个位置：患者自身的 32 维嵌入向量，掩码为 False (有效)
        vecs.append(emb[entity2id[ptid]])
        pad_masks.append(False) 
    else:
        # 极端兜底：患者极度缺乏特征未进入图谱，全零，掩码为 True (无效)
        vecs.append(torch.zeros(embed_dim))
        pad_masks.append(True) 
        
    # 第 1 到 34 个位置：全部用全 0 向量填充，掩码为 True (无效，被注意力机制屏蔽)
    for _ in range(1, max_seq_len):
        vecs.append(torch.zeros(embed_dim))
        pad_masks.append(True)
            
    # 返回知识序列矩阵和用于PyTorch注意力的掩码
    return torch.stack(vecs), torch.tensor(pad_masks, dtype=torch.bool)

def process_and_verify_seq(df, cohort_name):
    seq_tensors = []
    mask_tensors = []
    valid_count = 0
    
    for _, r in df.iterrows():
        seq_tensor, mask_tensor = get_patient_embedding_seq(r['filename'])
        seq_tensors.append(seq_tensor)
        mask_tensors.append(mask_tensor)
        
        # 验证是否成功提取到患者节点（第0个位置掩码为False）
        if not mask_tensor[0]:
            valid_count += 1
            
    stacked_seq = torch.stack(seq_tensors)
    stacked_mask = torch.stack(mask_tensors)
    
    print(f"[{cohort_name}] 总样本数: {len(df)} | 成功匹配CustomKG患者节点的样本数: {valid_count}")
    return stacked_seq, stacked_mask

# 执行数据提取
print("开始提取 CustomKG 患者节点序列与注意力掩码...")
ad_transe_seq, ad_transe_mask = process_and_verify_seq(pd.read_csv('AD.csv'), "AD")
mci_transe_seq, mci_transe_mask = process_and_verify_seq(pd.read_csv('mci.csv'), "MCI")
nc_transe_seq, nc_transe_mask = process_and_verify_seq(pd.read_csv('normal.csv'), "NC")

print("最终生成的知识序列 Shapes:", ad_transe_seq.shape, mci_transe_seq.shape, nc_transe_seq.shape)
print("最终生成的掩码张量 Shapes:", ad_transe_mask.shape, mci_transe_mask.shape, nc_transe_mask.shape)

In [9]:
# 第一步：合并原有的多模态特征 (EHR + IMG + BIO)
features_ad = torch.cat([ad_EHR, ad_output.cpu(), ad_tensor], dim=1)
features_mci = torch.cat([mci_EHR, mci_output.cpu(), mci_tensor], dim=1)
features_normal = torch.cat([normal_EHR, normal_output.cpu(), normal_tensor], dim=1)

# 第二步：沿着批次维度(dim=0)纵向合并所有样本的特征
features = torch.cat([features_ad, features_mci, features_normal], dim=0).float()

# 第三步：纵向合并上一步提取的所有样本的知识序列矩阵
kg_seq = torch.cat([ad_transe_seq, mci_transe_seq, nc_transe_seq], dim=0).float()

# 第四步：纵向合并对应的知识掩码矩阵
kg_mask = torch.cat([ad_transe_mask, mci_transe_mask, nc_transe_mask], dim=0).bool()

# 第五步：生成类别标签
y_ad = torch.ones(len(features_ad)) * 2  # AD 类别标签为 2
y_mci = torch.ones(len(features_mci)) * 1  # MCI 类别标签为 1
y_normal = torch.ones(len(features_normal)) * 0  # NC 类别标签为 0

# 分类任务的标签通常建议使用 long 类型
y = torch.cat([y_ad, y_mci, y_normal], dim=0).long() 

print("多模态特征 (features) 维度:", features.shape)
print("知识序列 (kg_seq) 维度:", kg_seq.shape)
print("注意力掩码 (kg_mask) 维度:", kg_mask.shape)
print("标签 (y) 维度:", y.shape)

多模态特征 (features) 维度: torch.Size([837, 117])
知识序列 (kg_seq) 维度: torch.Size([837, 35, 32])
注意力掩码 (kg_mask) 维度: torch.Size([837, 35])
标签 (y) 维度: torch.Size([837])


In [ ]:
from pathlib import Path
import sys
_split_root = next(parent for parent in (Path.cwd(), *Path.cwd().parents)
                   if (parent / "experiments" / "experiment_split.py").exists())
if str(_split_root) not in sys.path:
    sys.path.insert(0, str(_split_root))
from experiments.experiment_split import split_cohort_arrays, cohort_indices
# Fixed patient split: about 64% train, 16% validation, 20% test.
# Keep this split unchanged across model variants and all five training runs.

from pathlib import Path
import sys
repo_root = Path.cwd() if (Path.cwd() / "experiments" / "kg_ablation.py").exists() else Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from experiments.kg_ablation import sample_identity, run_ablation

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
import torch



# 1. 正常的划分数据集：同时切分特征(features), 图谱序列(kg_seq), 掩码(kg_mask) 和 标签(y)
# ★★★ 修复 NaN 报错的核心代码：防止全填充序列导致 Attention Softmax 除零崩溃 ★★★
# 找到那些掩码一整行全为 True 的样本（完全没有图谱节点的健康人）
all_padded_idx = kg_mask.all(dim=1)
# 强行把这些样本的第0个位置的掩码设为 False，使其能够去关注一个全零的 Padding 向量
kg_mask[all_padded_idx, 0] = False

record_ids, patient_ids = sample_identity([pd.read_csv("AD.csv"), pd.read_csv("mci.csv"), pd.read_csv("normal.csv")], ["AD", "MCI", "NC"], dataset="ADNI")
assert len(record_ids) == len(y)
X_train, X_val, X_test, seq_train, seq_val, seq_test, mask_train, mask_val, mask_test, y_train, y_val, y_test, record_train, record_val, record_test, patient_train, patient_val, patient_test = split_cohort_arrays(
    'ADNI', y.numpy(), features.detach().numpy(), kg_seq.detach().numpy(), kg_mask.detach().numpy(), y.numpy(), record_ids, patient_ids
)



# ----------------- 缺失值填充与归一化模块 -----------------
# 2. 初始化填充器和标准化器
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

# 3. 仅在训练集的“多模态特征(X)”上进行 fit_transform，严防数据泄露。图谱序列和掩码无需处理。
X_train = imputer.fit_transform(X_train)
X_train = scaler.fit_transform(X_train)

# 4. 在验证集和测试集上仅进行 transform
X_val = imputer.transform(X_val)
X_val = scaler.transform(X_val)

X_test = imputer.transform(X_test)
X_test = scaler.transform(X_test)
# ----------------------------------------------------

# 5. 转为 Tensor 并构建 DataLoader
X_train_tensor = torch.FloatTensor(X_train).to(device)
seq_train_tensor = torch.FloatTensor(seq_train).to(device)
mask_train_tensor = torch.BoolTensor(mask_train).to(device) # 注意掩码用 BoolTensor
y_train_tensor = torch.LongTensor(y_train).to(device)  

X_val_tensor = torch.FloatTensor(X_val).to(device)
seq_val_tensor = torch.FloatTensor(seq_val).to(device)
mask_val_tensor = torch.BoolTensor(mask_val).to(device)
y_val_tensor = torch.LongTensor(y_val).to(device)

X_test_tensor = torch.FloatTensor(X_test).to(device)
seq_test_tensor = torch.FloatTensor(seq_test).to(device)
mask_test_tensor = torch.BoolTensor(mask_test).to(device)
y_test_tensor = torch.LongTensor(y_test).to(device)

# 将 4 个张量打包进入 Dataset
train_dataset = TensorDataset(X_train_tensor, seq_train_tensor, mask_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, seq_val_tensor, mask_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, seq_test_tensor, mask_test_tensor, y_test_tensor)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [11]:
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_probs = []
    all_labels = []
    all_preds = []
    
    # 修改点：解包 4 个变量
    for inputs, kg_seq, kg_mask, labels in loader:
        inputs = inputs.to(device)
        kg_seq = kg_seq.to(device)
        kg_mask = kg_mask.to(device)
        labels = labels.to(device).long()
        
        optimizer.zero_grad()
        # 修改点：将特征、知识序列、掩码传入模型
        outputs = model(inputs, kg_seq, kg_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

        probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
        preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    train_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    train_f1 = f1_score(all_labels, all_preds, average='macro')
    train_recall = recall_score(all_labels, all_preds, average='macro')
    train_precision = precision_score(all_labels, all_preds, average='macro')

    avg_loss = total_loss / len(loader)
    return avg_loss, train_auc, train_f1, train_recall, train_precision

def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_probs = []
    all_labels = []
    all_preds = []
    with torch.no_grad():
        # 修改点：解包 4 个变量
        for inputs, kg_seq, kg_mask, labels in loader:
            inputs = inputs.to(device)
            kg_seq = kg_seq.to(device)
            kg_mask = kg_mask.to(device)
            labels = labels.to(device).long()
            
            # 修改点：将特征、知识序列、掩码传入模型
            outputs = model(inputs, kg_seq, kg_mask)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
            preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
    
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    val_auc = roc_auc_score(all_labels, all_probs, multi_class='ovo')
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    val_recall = recall_score(all_labels, all_preds, average='macro')
    val_precision = precision_score(all_labels, all_preds, average='macro')
    
    avg_loss = total_loss / len(loader)
    return avg_loss, val_auc, val_f1, val_recall, val_precision

In [ ]:
import torch.optim as optim
import torch.nn as nn

embed_dim        = 32
transe_embed_dim = 32           
num_epochs       = 200
batch_size       = 32
learning_rate    = 1e-5
weight_decay     = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 初始化模型 (调用模态感知检索版)

# Five predeclared seeds; each run uses the same split in all KG variants.
def make_model():
    return (
    KGMultiModalTransformer(
                embed_dim=embed_dim,
                transe_embed_dim=transe_embed_dim
            ).to(device)
    )

results, last_run = run_ablation(
    dataset="ADNI", graph="CustomKG", make_model=make_model,
    train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
    train_epoch=train_epoch, validate_epoch=validate_epoch,
    epochs=num_epochs, learning_rate=learning_rate,
    weight_decay=weight_decay, device=device,
    test_record_ids=record_test, test_patient_ids=patient_test,
    auc_mode="ovr",
)
model = last_run["model"]
all_labels = last_run["labels"]
all_probs = last_run["probabilities"]
preds = all_probs.argmax(axis=1)
train_losses = last_run["history"]["train_loss"]
train_aucs = last_run["history"]["train_auc"]
test_losses = last_run["history"]["val_loss"]  # plotted as validation
test_aucs = last_run["history"]["val_auc"]


In [ ]:
# Actual held-out test metrics for all five runs; no manual metric offsets.
import pandas as pd
summary = pd.DataFrame(results)
display(summary)
display(summary[["accuracy", "precision", "recall", "f1", "auc"]].agg(["mean", "std"]))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(range(len(train_aucs)), train_aucs, label="Train AUC", color="blue")
plt.plot(range(len(test_aucs)), test_aucs, label="Validation AUC", color="red")
plt.title("ADNI AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(range(len(train_losses)), train_losses, label="Train Loss", color="blue")
plt.plot(range(len(test_losses)), test_losses, label="Validation Loss", color="red")
plt.title("ADNI Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [24]:
from torchinfo import summary
import torch

# 1. 确保模型处于正确的设备上
model = model.to(device)

# 2. 构造与实际训练时维度相同的虚拟输入 (以 batch_size=32 为例)
# x 维度 = 14 (EHR) + 64 (img) + 39 (bio) = 117
dummy_x = torch.randn(32, 117).to(device)

# ★★★ 修改点：构造三维的知识序列和布尔型掩码 ★★★
# kg_seq 维度 = [batch_size, max_seq_len, embed_dim] = [32, 35, 128]
dummy_seq = torch.randn(32, 35, 128).to(device)

# kg_mask 维度 = [batch_size, max_seq_len] = [32, 35]，布尔型数据 (False代表有效节点)
dummy_mask = torch.zeros(32, 35, dtype=torch.bool).to(device)

# 3. 使用 torchinfo 打印模型详细结构和参数量
model_stats = summary(
    model, 
    input_data=(dummy_x, dummy_seq, dummy_mask), # ★ 修改点：传入三个虚拟参数
    col_names=["input_size", "output_size", "num_params", "trainable"],
    verbose=1
)

print(model_stats)

RuntimeError: Failed to run torchinfo. See above stack traces for more details. Executed layers up to: [Sequential: 1, Linear: 2, BatchNorm1d: 2, ReLU: 2, Dropout: 2, Linear: 2, BatchNorm1d: 2, ReLU: 2, Sequential: 1, Linear: 2, BatchNorm1d: 2, ReLU: 2, Dropout: 2, Linear: 2, BatchNorm1d: 2, ReLU: 2, Sequential: 1, Linear: 2, BatchNorm1d: 2, ReLU: 2, Dropout: 2, Linear: 2, BatchNorm1d: 2, ReLU: 2, Dropout: 2, Linear: 2, BatchNorm1d: 2, ReLU: 2]